# val_02_lickometer — validating lickometer licks against Lightning-Pose

A lick is detected in the pose data when the tongue comes within a threshold distance of either
spout. Detected licks are scored against the lickometer across three parameters: the spatial
threshold, a refractory filter, and the coincidence window used to call two events the same lick.

Session: `behavior_716325_2024-05-31_10-31-14`.

1. Setup
2. Load session data
3. Lick detection, worked example
4. Parameter sweep
5. Sweep figures
6. Inter-lick intervals
7. Metric curves at the chosen parameters
8. Individual events
9. Labeled video clips

Code Ocean only — needs the per-session `intermediate_data/` parquets and the labeled video.

## 1. Setup

In [ ]:
%matplotlib inline
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from aind_dynamic_foraging_behavior_video_analysis.kinematics.tongue_kinematics_utils import (
    mask_keypoint_data,
)
from aind_dynamic_foraging_behavior_video_analysis.kinematics.tongue_lickometer_utils import (
    detect_licks,
    filter_timestamps_refractory,
    calculate_metrics,
    calculate_metrics_witheventkeys,
    extract_clips_ffmpeg_encode,
)
from aind_dynamic_foraging_behavior_video_analysis.ephys.tongue_ephys import find_session_dir

In [ ]:
if Path("/root/capsule").exists():
    ENV     = "codeocean"
    SCRATCH = Path("/root/capsule/scratch")
    DATA    = Path("/root/capsule/data")
else:
    ENV     = "local"
    SCRATCH = Path("/Users/mib/Documents/Code/kinematics_analysis/data/for_local").parent
    DATA    = SCRATCH

IS_CO    = ENV == "codeocean"
FIG_DIR  = SCRATCH / "figures" / "val_02_lickometer"
SAVE_FIG = False

SESSION_DIR     = SCRATCH / "session_analysis_mlk"
EXAMPLE_SESSION = "behavior_716325_2024-05-31_10-31-14"
CONF            = 0.8      # masking confidence threshold

print("ENV={}".format(ENV))

## 2. Load session data

From the per-session `intermediate_data/` parquets written by
`tongue_analysis.run_batch_analysis`. `time_in_session` comes from `tongue_kins`, which
`kinematics_filter` returns with the same rows in the same order as the raw keypoints, and is
the same time base as `nwb_df_licks['timestamps']`.

`tongue_tip` is called `tongue_tip_center` in the current keypoint set.

In [ ]:
if not IS_CO:
    print("[skip] Code Ocean only: needs session_analysis_mlk/<session>/intermediate_data/.")
else:
    inter = find_session_dir(EXAMPLE_SESSION, roots=[SESSION_DIR]) / "intermediate_data"

    keypoint_dfs = {
        key: pd.read_parquet(inter / "kps_raw_{}.parquet".format(key))
        for key in ["tongue_tip_center", "spout_l", "spout_r"]
    }
    time_in_session = pd.read_parquet(
        inter / "tongue_kins.parquet", columns=["time_in_session"])["time_in_session"].values

    #extract tongue dataframe and mask
    tongue_masked = mask_keypoint_data(keypoint_dfs, "tongue_tip_center",
                                       confidence_threshold=CONF)
    # kps_raw_* already carries `time`, which is video-relative. Overwrite it with the
    # go-cue-relative session time, the base all_licks is on.
    tongue_masked["time"] = time_in_session

    #get all licks
    all_licks = np.sort(pd.read_parquet(inter / "nwb_df_licks.parquet")["timestamps"].to_numpy())

    #NB R and L are switched due to mislabeling in the training data
    mean_spoutL = np.mean(keypoint_dfs["spout_r"][["x", "y"]], 0)
    mean_spoutR = np.mean(keypoint_dfs["spout_l"][["x", "y"]], 0)

    print("Frames: {:,}   tracked at conf >= {}: {:,}".format(
        len(tongue_masked), CONF, int(tongue_masked["x"].notna().sum())))
    print("Lickometer licks: {:,}".format(len(all_licks)))

## 3. Lick detection, worked example

### 1. Detect licks in LP : use a threshold of distance to spout
![Defining a Lick](/root/capsule/scratch/figures/threshold_example_image.jpg)

### 2. Determine sensitivity and specificity of lickometer wrt LP licks
![Problems to overcome](/root/capsule/scratch/figures/examples_of_problems.jpg)

### 30 px spatial threshold, 0.05 s refractory filter, 0.1 s coincidence window

In [ ]:
#example: lickometer vs lightning pose validation


#Detect licks based on threshold crossing (pixels) close to either spout
LP_licks_temp = detect_licks(tongue_masked, mean_spoutL, mean_spoutR, 30) 
        
#filter licks that happen shortly after other licks (threshold fluctuation effect, 'problem 1' above)
LP_licks_temp = filter_timestamps_refractory(LP_licks_temp, 0.05)

#Calculate true positive, false positive, false negative with 0.1s overlap criteria ('problem 2' above)
tp, fp, fn = calculate_metrics(LP_licks_temp, all_licks, 0.1)

tp_rate = tp/len(all_licks)
fp_rate = fp/len(all_licks)
fn_rate = fn/len(LP_licks_temp)

#recall, true positive rate, sensitivity. of true events, how many are detected? recall = tp / (tp + fn)
recall = tp / (tp + fn)

#false negative rate, 1-recall. of true events, how many are not detected? false_negative_rate = fn / (tp + fn)
false_negative_rate = fn / (tp + fn)

#precision, positive predictive value. of detected events, how many are true? precision = tp / (tp + fp)
precision = tp / (tp + fp)

#false discovery rate, 1-precision. of detected events, how many are not true? false_discovery_rate = fp / (tp + fp)
false_discovery_rate = fp / (tp + fp)

print(f'false_negative_rate: {false_negative_rate:.2f}')
print(f'false_discovery_rate: {false_discovery_rate:.2f}')

## 4. Parameter sweep

In [ ]:
# calculate false positive and false negative rate across range of parameter space

# Parameters to test
#1. spatial thresholds for lick detection (in pixels)
#2. time thresholds for overlap between LP and lickometer coincidence (ie whether 'same event' or not)
#3. time thresholds for 'refractory filter' -- removes spurious detection of licks due to oscillation around spatial lick detection threshold
spatial_thresholds = np.arange(10, 51, 5)
time_thresholds = np.arange(0.005, 0.251, 0.005)
t_refractory_values = np.arange(0, 0.11, 0.01)  
relevant_licks_temp = all_licks

# Initialize a list to collect results
results = []

# Loop through spatial thresholds
for spatial_threshold in spatial_thresholds:
    # Detect licks based on spatial threshold
    LP_licks_temp = detect_licks(tongue_masked, mean_spoutL, mean_spoutR, spatial_threshold)

    # Loop through refractory values
    for t_refractory in t_refractory_values:
        # Filter detected licks based on the current t_refractory value
        LP_licks_filtered = filter_timestamps_refractory(LP_licks_temp, t_refractory)

        # Loop through time thresholds
        for time_threshold in time_thresholds:
            tp, fp, fn = calculate_metrics(LP_licks_filtered, relevant_licks_temp, time_threshold)
            
            # Calculate additional metrics on top
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            false_negative_rate = fn / (tp + fn) if (tp + fn) > 0 else 0
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            false_discovery_rate = fp / (tp + fp) if (tp + fp) > 0 else 0
            f1_score = (2*tp) / (2*tp + fp + fn) #harmonic mean of precision and recall
            
            
            # Append results to the list
            results.append({
                'spatial_threshold': spatial_threshold,
                'time_threshold': time_threshold,
                't_refractory': t_refractory,
                'true_positive': tp,
                'false_positive': fp,
                'false_negative': fn,
                'recall': recall,
                'false_negative_rate': false_negative_rate,
                'precision': precision,
                'false_discovery_rate': false_discovery_rate,
                'f1_score': f1_score
            })

# Convert results to a DataFrame
results_df = pd.DataFrame(results)

In [ ]:
# What are the top results?
n = 5
test_df = results_df
# # modify below to select subset of parameters, eg
# test_df = results_df.query('time_threshold <= 0.1 and t_refractory <= 0.05')

# Get the top N rows with the highest recall values
top_n_recall_rows = test_df.nlargest(n, 'f1_score')

# Display the top N rows
top_n_recall_rows

## 5. Sweep figures

In [ ]:
#Plotting results across parameters

#figure 1: heatmaps comparing each parameter (time treshold, spatial threshold, refractory filter time)

df = results_df.copy()
df['time_threshold'] = df['time_threshold'].round(3) #due to plotting issues with floats

# Define the color range for the heatmap
cmap_range_min = 0.5  # Set this to the minimum score you want to visualize
cmap_range_max = 1  # Set this to the maximum score you want to visualize

# Create a figure with three subplots
fig, axs = plt.subplots(1, 3, figsize=(16, 4))

# Heatmap 1: spatial_threshold vs. time_threshold
heatmap_data1 = df.pivot_table(values='f1_score', index='spatial_threshold', columns='time_threshold')
sns.heatmap(heatmap_data1, ax=axs[0], cmap='YlGnBu', cbar_kws={'label': 'F1 Score'}, vmin=cmap_range_min, vmax=cmap_range_max)

# Find the max value and its position for heatmap 1
max_value1 = heatmap_data1.max().max()
max_pos1 = heatmap_data1.stack().idxmax()
spatial_threshold1, time_threshold1 = max_pos1
print(f"Heatmap 1 Max - Spatial Threshold: {spatial_threshold1}, Time Threshold: {time_threshold1}, F1 Score: {max_value1:.2f}")

axs[0].set_title('Spatial Threshold vs. Time Threshold')
axs[0].set_xlabel('Time Threshold')
axs[0].set_ylabel('Spatial Threshold')

# Heatmap 2: spatial_threshold vs. t_refractory
heatmap_data2 = df.pivot_table(values='f1_score', index='spatial_threshold', columns='t_refractory')
sns.heatmap(heatmap_data2, ax=axs[1], cmap='YlGnBu', cbar_kws={'label': 'F1 Score'}, vmin=cmap_range_min, vmax=cmap_range_max)

# Find the max value and its position for heatmap 2
max_value2 = heatmap_data2.max().max()
max_pos2 = heatmap_data2.stack().idxmax()
spatial_threshold2, t_refractory2 = max_pos2
print(f"Heatmap 2 Max - Spatial Threshold: {spatial_threshold2}, T Refractory: {t_refractory2}, F1 Score: {max_value2:.2f}")

axs[1].set_title('Spatial Threshold vs. T Refractory')
axs[1].set_xlabel('T Refractory')
axs[1].set_ylabel('Spatial Threshold')

# Heatmap 3: time_threshold vs. t_refractory
heatmap_data3 = df.pivot_table(values='f1_score', index='time_threshold', columns='t_refractory')
sns.heatmap(heatmap_data3, ax=axs[2], cmap='YlGnBu', cbar_kws={'label': 'F1 Score'}, vmin=cmap_range_min, vmax=cmap_range_max)

# Find the max value and its position for heatmap 3
max_value3 = heatmap_data3.max().max()
max_pos3 = heatmap_data3.stack().idxmax()
time_threshold3, t_refractory3 = max_pos3
print(f"Heatmap 3 Max - Time Threshold: {time_threshold3}, T Refractory: {t_refractory3}, F1 Score: {max_value3:.2f}")

axs[2].set_title('Time Threshold vs. T Refractory')
axs[2].set_xlabel('T Refractory')
axs[2].set_ylabel('Time Threshold')

# Adjust layout and show the plot
plt.tight_layout()
plt.show()



# Figure 2: interaction plots for false negative rate, false discovery rate, and f1 score for time threshold and spatial threshold

# Create interaction plot with subplots
fig, axs = plt.subplots(1, 3, figsize=(16, 4))

# Group the DataFrame by 'time_threshold' and 'spatial_threshold' and calculate the mean of the relevant columns
grouped_df = results_df.groupby(['time_threshold', 'spatial_threshold']).mean().reset_index()

# Get unique spatial thresholds and define a colormap
unique_spatial_thresholds = grouped_df['spatial_threshold'].unique()
num_colors = len(unique_spatial_thresholds)
cmap = plt.get_cmap('viridis', num_colors)  # Use a colormap with enough distinct colors

# FNR interaction plot
for i, spatial_threshold in enumerate(unique_spatial_thresholds):
    # Filter DataFrame for the current spatial threshold
    filtered_df = grouped_df[grouped_df['spatial_threshold'] == spatial_threshold]
    
    # Plot the False Negative Rate
    axs[0].plot(filtered_df['time_threshold'], 
                 filtered_df['false_negative_rate'], 
                 label=f'Spatial: {spatial_threshold}', 
                 color=cmap(i))

axs[0].set_title('False Negative Rate')
axs[0].set_xlabel('Time Threshold (s)')
axs[0].set_ylabel('False Negative Rate: fn / (tp + fn)')
# axs[0].legend(title='Spatial Threshold (pixels)', bbox_to_anchor=(1.05, 1), loc='upper left')
axs[0].grid()

# FDR interaction plot
for i, spatial_threshold in enumerate(unique_spatial_thresholds):
    # Filter DataFrame for the current spatial threshold
    filtered_df = grouped_df[grouped_df['spatial_threshold'] == spatial_threshold]
    
    # Plot the False Discovery Rate
    axs[1].plot(filtered_df['time_threshold'], 
                 filtered_df['false_discovery_rate'], 
                 label=f'Spatial: {spatial_threshold}', 
                 color=cmap(i))

axs[1].set_title('False Discovery Rate')
axs[1].set_xlabel('Time Threshold (s)')
axs[1].set_ylabel('False Discovery Rate: fp / (tp + fp)')
# axs[1].legend(title='Spatial Threshold (pixels)', bbox_to_anchor=(1.05, 1), loc='upper left')
axs[1].grid()

# f1_score interaction plot
for i, spatial_threshold in enumerate(unique_spatial_thresholds):
    # Filter DataFrame for the current spatial threshold
    filtered_df = grouped_df[grouped_df['spatial_threshold'] == spatial_threshold]
    
    # Plot the False Discovery Rate
    axs[2].plot(filtered_df['time_threshold'], 
                 filtered_df['f1_score'], 
                 label=f'Spatial: {spatial_threshold}', 
                 color=cmap(i))

axs[2].set_title('F1 Score')
axs[2].set_xlabel('Time Threshold (s)')
axs[2].set_ylabel('F1 score = (2*tp) / (2*tp + fp + fn)')
axs[2].legend(title='Spatial Threshold (pixels)', bbox_to_anchor=(1.05, 1), loc='upper left')
axs[2].grid()

fig.suptitle('Interaction of Spatial Threshold and Time Threshold', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
#Optional: heatmap facet grid of f1 score, comparing spatial threshold and time threshold for each t_refractory value

# Create a FacetGrid for f1 score
g = sns.FacetGrid(results_df, col="t_refractory", col_wrap=3, height=3, aspect=1)

# Loop through each unique t_refractory value
cbar_bool = False
for k, t_refractory in enumerate(results_df['t_refractory'].unique()):
    # Filter the DataFrame for the current t_refractory value
    filtered_df = results_df[results_df['t_refractory'] == t_refractory]
    filtered_df['time_threshold'] = filtered_df['time_threshold'].round(3)

    # Pivot the filtered data for recall
    pivot_data = filtered_df.pivot_table(index="spatial_threshold", 
                                          columns="time_threshold", 
                                          values="f1_score", 
                                          aggfunc='mean')

    # Create the heatmap for recall without a colorbar
    ax = g.axes[k]

    if k == len(results_df['t_refractory'].unique()) - 1:
        cbar_bool = True

    sns.heatmap(pivot_data, ax=ax, cmap='YlGnBu', 
                annot=False, linewidths=.5,
                vmin=0.8, vmax=1.0, 
                cbar=cbar_bool,
                cbar_kws={'label': 'f1_score'})  

    # Set title for the subplot
    ax.set_title(f'Refractory Time: {t_refractory:.2f} s')




# Set the overall axis labels
g.set_axis_labels('Time Threshold (s)', 'Spatial Threshold (pixels)')
g.set_titles(col_template='Refractory Time: {col_name:.2f} s')

plt.subplots_adjust(top=0.9)
plt.subplots_adjust(wspace=0.1, hspace=0.3) 
g.fig.suptitle('F1 Score Heatmaps by Spatial Threshold and Time Threshold', fontsize=12)

plt.show()

### Interpretation

- conclusions: unbiased search for best parameter would indicate pixel_threshold of 35.
- also indicates maximizing both the refractory filter value, and the overlap threshold value, produces best performance
- however, unsure if this is truly warranted -- we want to avoid detecting two licks as 'the same' when in fact they are not.
- --> pursue data-driven parameter selection

## 6. Inter-lick intervals

In [ ]:
# let's motivate using the refractory filter by analyzing inter lick interval distribution
from matplotlib import cm
import matplotlib.patches as patches

# plt.figure(figsize=(10,6))

fig, axs = plt.subplots(1, 2, figsize=(8, 4),sharey=False)


cmap = cm.YlGnBu(np.linspace(0, 1, len(spatial_thresholds)))

# Loop through spatial thresholds
for i, spatial_threshold in enumerate(spatial_thresholds):
    # Detect licks based on spatial threshold
    LP_licks_temp = detect_licks(tongue_masked, mean_spoutL, mean_spoutR, spatial_threshold)

    ILIs = np.diff(LP_licks_temp)
    # Calculate and plot the CDF
    sorted_ILIs = np.sort(ILIs)
    cdf = np.arange(1, len(sorted_ILIs) + 1) / len(sorted_ILIs)
    axs[0].plot(sorted_ILIs, cdf, label=f'Threshold: {spatial_threshold:.2f}', color=cmap[i], alpha=0.7)
    axs[1].plot(sorted_ILIs, cdf, label=f'Threshold: {spatial_threshold:.2f}', color=cmap[i], alpha=0.7)


# Set x-axis limits
axs[0].set_xlim(0, 5)
axs[0].set_ylim(0,1)
axs[1].set_xlim(0,0.5)
axs[1].set_ylim(0,0.8)


# Customize the plot
fig.suptitle('CDF of Inter-Lick Intervals Across Spatial Thresholds')
# plt.xlabel('Inter-Lick Interval (seconds)')
# plt.ylabel('Cumulative Probability')

fig.text(0.5, 0.04, 'Inter-Lick Interval (seconds)', ha='center')
fig.text(0.04, 0.5, 'Cumulative Probability', va='center', rotation='vertical')


rect = patches.Rectangle((0,0), 0.5, 0.8, linewidth=2, edgecolor='r', facecolor='none')
# Add the patch to the axes
axs[0].add_patch(rect)

axs[0].legend()
axs[0].grid()
axs[1].grid()

axs[1].spines['bottom'].set_color('red')
axs[1].spines['top'].set_color('red') 
axs[1].spines['right'].set_color('red')
axs[1].spines['left'].set_color('red')

plt.show()

- each spatial threshold results in some ILIs that are less than 100ms, faster than a lick can occur.
- let's compare to the lickometer event times.
- from before, we know we want to use threshold of 35 pixels

In [ ]:
spatial_threshold = 35
LP_licks = detect_licks(tongue_masked, mean_spoutL, mean_spoutR, spatial_threshold)

plt.figure(figsize=(6,4))
LP_lick_diffs = np.diff(LP_licks)
plt.hist(LP_lick_diffs,bins=100,range=[0,.5],alpha=0.8,density=True)
all_lick_diffs = np.diff(all_licks)
plt.hist(all_lick_diffs,bins=100,range=[0,.5],alpha=0.8,density=True)
plt.title(f'Histogram of ILIs: 35 pixel threshold')
plt.xlabel('Time (s)')
plt.ylabel('Count')
plt.legend(['Lightning Pose','Lickometer'])
plt.show()

spatial_threshold = 30
LP_licks = detect_licks(tongue_masked, mean_spoutL, mean_spoutR, spatial_threshold)

plt.figure(figsize=(6,4))
LP_lick_diffs = np.diff(LP_licks)
plt.hist(LP_lick_diffs,bins=100,range=[0,.5],alpha=0.8,density=True)
all_lick_diffs = np.diff(all_licks)
plt.hist(all_lick_diffs,bins=100,range=[0,.5],alpha=0.8,density=True)
plt.title(f'Histogram of ILIs: 30 pixel threshold')
plt.xlabel('Time (s)')
plt.ylabel('Count')
plt.legend(['Lightning Pose','Lickometer'])
plt.show()

## 7. Metric curves at the chosen parameters

- this seems to motivate filter of licks within 100ms of another lick.
- data seem cleaner for threshold of 30 pixels -- I wonder if ILIs around 100ms are being shifted 'longer', as they need to get closer to the spout.
- --> would imply these are real events -- and by moving the threshold out further from mouth (closer to spout), we are gaining separation between 'real' licks and licks identified due to oscillation around the threshold
- motivates combination of 30 pixel, 100ms refractory filter.
- Now, let's calculate basic metric curves with these parameters

In [ ]:
# Now, let's calculate basic metric curves with these parameters


# Set specific values for spatial threshold and refractory period
spatial_threshold = 30
t_refractory = 0.1
time_thresholds = np.arange(0.005, 0.251, 0.001)  
relevant_licks_temp = all_licks

# Initialize a list to collect results
results = []

# Detect licks based on the fixed spatial threshold
LP_licks_temp = detect_licks(tongue_masked, mean_spoutL, mean_spoutR, spatial_threshold)

# Filter detected licks based on the fixed t_refractory value
LP_licks_filtered = filter_timestamps_refractory(LP_licks_temp, t_refractory)

# Loop through time thresholds
for time_threshold in time_thresholds:
    tp, fp, fn = calculate_metrics(LP_licks_filtered, relevant_licks_temp, time_threshold)
    
    # Calculate additional metrics
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    false_negative_rate = fn / (tp + fn) if (tp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    false_discovery_rate = fp / (tp + fp) if (tp + fp) > 0 else 0
    f1_score = (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0  # harmonic mean of precision and recall
    
    # Append results to the list
    results.append({
        'spatial_threshold': spatial_threshold,
        'time_threshold': time_threshold,
        't_refractory': t_refractory,
        'true_positive': tp,
        'false_positive': fp,
        'false_negative': fn,
        'recall': recall,
        'false_negative_rate': false_negative_rate,
        'precision': precision,
        'false_discovery_rate': false_discovery_rate,
        'f1_score': f1_score
    })

# Convert results to a DataFrame
results_parameterized_df = pd.DataFrame(results)

In [ ]:
# Extract relevant data from the DataFrame
time_thresholds = results_parameterized_df['time_threshold']
false_negative_rate = results_parameterized_df['false_negative_rate']
false_discovery_rate = results_parameterized_df['false_discovery_rate']
f1_score = results_parameterized_df['f1_score']

# Create a figure and axis
fig, ax1 = plt.subplots()

# Plot false_negative_rate and false_discovery_rate on the first axis
ax1.plot(time_thresholds, false_negative_rate, label='False Negative Rate', color='b')
ax1.plot(time_thresholds, false_discovery_rate, label='False Discovery Rate', color='r')
ax1.plot(time_thresholds, f1_score, label='F1 Score', color='g')  # Added F1 Score to the same plot

# Set labels
ax1.set_xlabel('Time Threshold')
ax1.set_ylabel('Rate', color='k')
ax1.tick_params(axis='y', labelcolor='k')

# Add legend
ax1.legend(loc='center right')

# Add a title and grid
plt.title('Pixel Threshold: 30 | Refractory Filter: 0.1 s')
ax1.grid()

# Show the plot
plt.show()

### Final stats

- 100 ms seems like a safe time overlap threshold --> not getting any further improvement in F1 score
- final stats:

In [ ]:
chosen_row = results_parameterized_df.query('time_threshold == 0.1')
chosen_row

## 8. Individual events

`calculate_metrics_witheventkeys(a, b, w)` names its parameters `ground_truth,
detected_events`, and the pose events are passed first. So `fn` counts unmatched pose events
and `fp` counts unmatched lickometer events: `FP_times` below is taken from the lickometer
frame, and the clips in §9 are written to a directory named `false_positive/`.

In [ ]:
# Now pull out individual events (false positives, false negatives) and analyze



#use parameters from above:

# Detect licks based on the fixed spatial threshold
LP_licks = detect_licks(tongue_masked, mean_spoutL, mean_spoutR, 30)
# Filter detected licks based on the fixed t_refractory value
LP_licks = filter_timestamps_refractory(LP_licks, 0.1)
all_licks = filter_timestamps_refractory(all_licks, 0.1)


tp, fp, fn, LP_licks_classified, licko_licks_classified = calculate_metrics_witheventkeys(LP_licks, all_licks, time_window = 0.1)

In [ ]:
#calculate distance to nearest spout

spoutL_pos = np.array([mean_spoutL['x'], mean_spoutL['y']])
spoutR_pos = np.array([mean_spoutR['x'], mean_spoutR['y']])

#Create a mask to filter out rows with NaNs in 'x' and 'y'
mask = tongue_masked[['x', 'y']].notna().all(axis=1)

#Calculate distances only for valid rows
valid_rows = tongue_masked[mask]
distances_L = np.linalg.norm(valid_rows[['x', 'y']].to_numpy() - spoutL_pos, axis=1)
distances_R = np.linalg.norm(valid_rows[['x', 'y']].to_numpy() - spoutR_pos, axis=1)

#Insert the calculated distances back into the original DataFrame
tongue_masked.loc[mask, 'distance_to_left_spout'] = distances_L
tongue_masked.loc[mask, 'distance_to_right_spout'] = distances_R

# Determine the nearest spout for each timepoint
tongue_masked['nearest_spout_distance'] = tongue_masked[['distance_to_left_spout', 'distance_to_right_spout']].min(axis=1)

# Optionally, add a column to indicate which spout is the nearest
tongue_masked['nearest_spout'] = tongue_masked[['distance_to_left_spout', 'distance_to_right_spout']].idxmin(axis=1,skipna=True).apply(lambda x: 'Left' if x == 'distance_to_left_spout' else 'Right')

# Drop the individual distances if you only need the nearest spout distance
tongue_masked = tongue_masked.drop(columns=['distance_to_left_spout', 'distance_to_right_spout'])

In [ ]:
def plot_tongue_trajectory(dataframe, event_times, clip_length, event_type):
    # Check if event times exceed the limit
    if len(event_times) > 50:
        raise ValueError("Cannot plot more than 50 events at once.")

    # Loop through each event time and create a plot
    for i, event_time in enumerate(event_times):
        # Center the timestamps for the event
        timestamps_centered = event_time - clip_length / 2

        # Create the plots
        fig, axs = plt.subplots(3, 1, figsize=(4, 4), sharex=True)

        # Plot the tongue trajectory
        line1, = axs[0].plot(dataframe['time'], dataframe['y'], linestyle='-', color='b', label='Kinematic Trajectory')
        axs[0].set_ylabel('Y Position')
        axs[0].set_ylim([200, 400])

        # Plot the X Position (no label needed here for legend)
        axs[1].plot(dataframe['time'], dataframe['x'], linestyle='-', color='b')
        axs[1].set_ylabel('X Position')
        axs[1].set_ylim([300, 400])

        # Plot the distance to the nearest spout
        line2, = axs[2].plot(dataframe['time'], dataframe['nearest_spout_distance'], linestyle='-', color='g', label='Distance to Spout')
        axs[2].set_xlabel('Time')
        axs[2].set_ylabel('Distance to Spout')

        xlim_inx = [timestamps_centered, timestamps_centered + clip_length]

        axs[2].set_xlim(xlim_inx)
        axs[2].set_ylim([0, 55])
        line3 = axs[2].axhline(y=30, color='k', ls='--', lw=0.5, label='Spout Threshold')
        line4 = axs[2].axvline(x=xlim_inx[0] + (clip_length / 2), color='r', ls='--', lw=0.5, label=f'Time of {event_type}')

        # Create a combined legend for the entire figure
        handles = [line1, line2, line3, line4]
        labels = [line.get_label() for line in handles]
        fig.legend(handles, labels, bbox_to_anchor=(1.05, .65), loc='upper left')

        # Adjust layout and show the plots
        plt.tight_layout()
        plt.show()

FP_times = licko_licks_classified[licko_licks_classified['Status'] == "False Positive"]["Time"].to_numpy()
FN_times = LP_licks_classified[LP_licks_classified['Status'] == "False Negative"]["Time"].to_numpy()

# Call the function for False Negatives
# plot_tongue_trajectory(tongue_masked, FN_times[:2], clip_length=1.0, event_type='False Negative')

# Call the function for False Positives 
plot_tongue_trajectory(tongue_masked, FP_times[:2], clip_length=1.0, event_type='False Positive')

## 9. Labeled video clips

In [ ]:
#clip labeled video at example timepoints

# clip videos to visualize failure modes
timestamps = FP_times[0:2]
input_video_path = str(DATA / 'video_preds_labeltest/labeled_videos/bottom_camera_labeled.mp4')
clip_length = 1.0 # Clip length in seconds
timestamps_centered  = timestamps - clip_length/2
output_dir = str(SCRATCH / 'labeled_clips' / 'false_positive')


import subprocess
extract_clips_ffmpeg_encode(input_video_path, timestamps_centered, clip_length, output_dir)